<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [2]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [3]:
# Select one of our countries to work on...
iso_code = "136"
iso_country = "Cayman Islands"
year = 2024
gdp = 7.770e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [4]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Cayman Islands': 

Number of trade records present in dataset: 5542
Number of partner countries present in dataset: 65
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1228.8m
Total weight within the dataset: 1596.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,136,Cayman Isds,124,Canada,Import,Sea,830990,830990 | Stoppers; caps and l,8309,"8309 | Stoppers, caps, lids",83,83 | Metal; miscellaneous,42932,"Stoppers, caps and lids (including crown corks...",Metal Products,37,FMP - Metal products,28069.776,NaN,28069.776,kg,3656.703,3656.703,5745.883147,4282.0,2660.710622,15288.132323,['Container'],['Container'],79.908456,79.768934,79.782569,79.782569,79.845091,79.845091,83.588308,148.144707,92.200583,85.599662,85.832131,87.981870,81.847635,145.085055,97.046928,88.893536,86.853605,90.787209,80.320979,157.770164,120.342848,105.24761,97.373961,94.627160,78.822607,167.748347,138.037624,129.352969,104.251355,101.416636,77.352518,169.989452,162.409879,163.806292,113.019300,106.534907,75.852961,166.495499,159.553698,174.814520,129.309096,108.487528,74.440270,148.518025,154.795315,162.209337,121.597247,109.769363,72.999776,136.426499,138.681106,172.986781,142.911354,110.496264,42932
1,1,136,Cayman Isds,124,Canada,Import,Sea,841381,841381 | Pumps and liquid ele,8413,"8413 | Pumps; for liquids,",84,84 | Machinery and mechan,4322,Pumps for liquids; liquid elevators,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,96356.182,NaN,96356.182,NaN,0.000,2747.444,5733.192829,4282.0,2660.710622,15254.367058,['Container'],['Container'],79.731971,79.592757,79.606362,79.606362,79.668746,79.668746,83.403696,147.817516,91.996950,85.410607,85.642563,87.787554,81.666867,144.764622,96.832591,88.697206,86.661781,90.586697,80.143583,157.421714,120.077060,105.01516,97.158902,94.418167,78.648520,167.377860,137.732755,129.067281,104.021106,101.192648,77.181677,169.614015,162.051182,163.444511,112.769687,106.299614,75.685432,166.127778,159.201309,174.428426,129.023505,108.247923,74.275862,148.190009,154.453435,161.851083,121.328688,109.526928,72.838549,136.125189,138.374815,172.604723,142.595721,110.252222,4322


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $316.4m (25.7%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
22 | Beverages, spirits a  53930443.3  475528552.9      4.4     29.8
02 | Meat and edible meat  44971261.7   15760798.8      3.7      1.0
21 | Miscellaneous edible  44842308.9   10899178.4      3.6      0.7
07 | Vegetables and certa  36152706.7   38360401.2      2.9      2.4
04 | Dairy produce; birds  28050686.4   13408858.1      2.3      0.8
08 | Fruit and nuts, edib  19543049.1   16624585.3      1.6      1.0
19 | Preparations of cere  18599801.4    6762410.1      1.5      0.4
20 | Preparations of vege  11267581.2    8248422.3      0.9      0.5
16 | Meat, fish, crustace  11187295.8    2450086.8      0.9      0.2
03 | Fish and crustaceans  10249378.6    2318547.6      0.8      0.1
23 | Food industries, res   9971484.1    7693539.9      0.8      0.5
09

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [6]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $297.0m (24.2%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  159130248.6  513421336.1     12.9     32.2
Agriculture       134389406.5   89549249.9     10.9      5.6
Fishing             3438624.7     532268.2      0.3      0.0



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $240.6m (19.6%), ... 

                                  cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                               
OFD - Food products n.e.c.     108211512.9  48988524.2      8.8      3.1
V_F - Vegetables, fruit, nuts   35560773.0  37129736.8      2.9      2.3
OMT - Meat products n.e.c.      35053077.3  14575365.9      2.9      0.9
MIL - Dairy products            26005117.5  11838019.0      2.1      0.7
CMT - Bovine meat prods         19088870.9   3156565.4      1.6      0.2
OCR - Crops n.e.c.               7384994.9   2519012.9      0.6      0.2
VOL - Vegetable oils and fats    3809196.0   2215560.4      0.3      0.1
PCR - Processed rice             2118865.9   3680470.1      0.2      0.2
SGR - Sugar                      1008592.7   2286207.5      0.1      0.1
PDR - Paddy rice                  880082.5   1528703.2      0.1      0.1
GRO - Cereal grains n.e.c.        802919.1   1531244.5    

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 25.7% 
GTAP 19.6% 
EORA 24.2%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_16126/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,136,Cayman Isds,124,Canada,Import,Sea,830990,830990 | Stoppers; caps and l,8309,"8309 | Stoppers, caps, lids",83,83 | Metal; miscellaneous,42932,"Stoppers, caps and lids (including crown corks...",Metal Products,37,FMP - Metal products,28069.776,NaN,28069.776,kg,3656.703,3656.703,5745.883147,4282.0,2660.710622,15288.132323,['Container'],['Container'],79.908456,79.768934,79.782569,79.782569,79.845091,79.845091,83.588308,148.144707,92.200583,85.599662,85.832131,87.981870,81.847635,145.085055,97.046928,88.893536,86.853605,90.787209,80.320979,157.770164,120.342848,105.24761,97.373961,94.627160,78.822607,167.748347,138.037624,129.352969,104.251355,101.416636,77.352518,169.989452,162.409879,163.806292,113.019300,106.534907,75.852961,166.495499,159.553698,174.814520,129.309096,108.487528,74.440270,148.518025,154.795315,162.209337,121.597247,109.769363,72.999776,136.426499,138.681106,172.986781,142.911354,110.496264,42932,NaN
1,1,136,Cayman Isds,124,Canada,Import,Sea,841381,841381 | Pumps and liquid ele,8413,"8413 | Pumps; for liquids,",84,84 | Machinery and mechan,4322,Pumps for liquids; liquid elevators,Electrical and Machinery,41,OME - Machinery and equipment n.e.c.,96356.182,NaN,96356.182,NaN,0.000,2747.444,5733.192829,4282.0,2660.710622,15254.367058,['Container'],['Container'],79.731971,79.592757,79.606362,79.606362,79.668746,79.668746,83.403696,147.817516,91.996950,85.410607,85.642563,87.787554,81.666867,144.764622,96.832591,88.697206,86.661781,90.586697,80.143583,157.421714,120.077060,105.01516,97.158902,94.418167,78.648520,167.377860,137.732755,129.067281,104.021106,101.192648,77.181677,169.614015,162.051182,163.444511,112.769687,106.299614,75.685432,166.127778,159.201309,174.428426,129.023505,108.247923,74.275862,148.190009,154.453435,161.851083,121.328688,109.526928,72.838549,136.125189,138.374815,172.604723,142.595721,110.252222,4322,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $202.4m of $1228.8m (16.5%)
Volume: 124.3 of 1596.6 million kilograms (7.8%)
Value-to-Weight: 1.63 from 0.77 US$ per kg (+111.6%)
Average distance to market (volume-weighted): 963.0 from 962.9 nm (+0.0%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    220  69560985.5  28882337.5      5.7      1.8
1           Vegetables    157  43218062.3  44202880.9      3.5      2.8
2        Meat and Prep     46  38766427.5  14471618.3      3.2      0.9
3                Dairy     61  23717795.2  11131595.7      1.9      0.7
4               Fruits     93  18782945.1  16242894.7      1.5      1.0
5      Sugar and Honey     42   4398560.9   5321065.9      0.4      0.3
6        Fats and Oils     45   2946185.7   1743565.4      0.2      0.1
7  Unprocessed Cereals     23   1045859.6   2316211.7      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [12]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,4884606.1,8410079.4,5329946.1,4968608.4,4981888.7,5118732.6,4542557.2,9831681.4,9327413.7,9446597.0,6449768.8,6134135.6,4301574.0,7826117.5,7923131.6,9636740.9,8175413.6,6352482.8
1,Fats and Oils,829634.3,1409914.0,901906.2,841379.2,843585.6,867768.2,773478.6,1610386.6,1549429.4,1578692.1,1091136.6,1033109.0,734204.6,1311519.1,1330688.9,1609871.1,1346764.5,1068354.1
2,Fruits,8268890.1,14601652.8,9109610.5,8461695.6,8484509.4,8698989.2,7657058.3,16754257.0,16012071.9,16162125.8,11153064.7,10518173.1,7229891.2,13461273.9,13683338.3,17022529.3,14078141.9,10906609.8
3,Meat and Prep,7086348.1,12559232.2,7816469.1,7256864.2,7276572.2,7458820.1,6557697.8,14411159.4,13768587.6,13886971.0,9581413.0,9031687.0,6188686.4,11565800.1,11756938.4,14665263.3,12115565.3,9367518.1
4,Processed Cereals,13384432.2,22856135.9,14567442.9,13596414.8,13632513.1,14013045.6,12465964.8,26275320.9,25171026.3,25632007.6,17722889.1,16717403.7,11823696.4,21337537.1,21740746.6,26282927.3,21936474.4,17290135.5
5,Sugar and Honey,2031663.2,3481626.7,2211584.8,2063238.4,2069065.1,2127729.0,1890872.3,4011763.9,3840539.3,3914148.6,2704945.0,2541225.4,1792458.6,3252814.6,3320447.5,4012308.1,3345446.5,2628142.8
6,Unprocessed Cereals,1058511.7,1704306.6,1129934.3,1063083.8,1065636.8,1099431.4,995566.3,1960498.0,1887277.4,1948028.9,1350901.2,1281944.7,951366.9,1622285.6,1654061.2,1905909.9,1620886.8,1320425.6
7,Vegetables,21358000.1,37421640.5,23463734.5,21820192.0,21878342.3,22442947.5,19805870.5,42953037.9,41068320.4,41525312.4,28659063.9,27052877.0,18720327.1,34582045.1,35155053.5,43465218.9,36022473.2,28036752.0


### 3.3.2 Total FEM costs by scenario (% cif)

In [13]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.15,0.02,0.00,0.00,0.01,0.22,0.20,0.21,0.08,0.07,0.15,0.15,0.22,0.16,0.09
1,Fats and Oils,0.20,0.02,0.00,0.00,0.01,0.28,0.26,0.27,0.11,0.09,0.20,0.20,0.30,0.21,0.11
2,Fruits,0.34,0.04,0.01,0.01,0.02,0.48,0.44,0.45,0.19,0.15,0.33,0.34,0.52,0.36,0.20
3,Meat and Prep,0.14,0.02,0.00,0.00,0.01,0.20,0.19,0.19,0.08,0.06,0.14,0.14,0.22,0.15,0.08
4,Processed Cereals,0.14,0.02,0.00,0.00,0.01,0.20,0.18,0.19,0.08,0.06,0.14,0.14,0.21,0.15,0.08
5,Sugar and Honey,0.33,0.04,0.01,0.01,0.02,0.48,0.44,0.46,0.19,0.15,0.33,0.35,0.50,0.35,0.19
6,Unprocessed Cereals,0.62,0.07,0.00,0.01,0.04,0.92,0.85,0.91,0.34,0.27,0.64,0.67,0.91,0.64,0.35
7,Vegetables,0.37,0.05,0.01,0.01,0.03,0.54,0.49,0.50,0.20,0.17,0.37,0.38,0.57,0.40,0.22
